# Signed Jumps 04 — jumps at 1-second resolution

Protocol: `discussions/Signed_Jumps/007_claude_sj04_preregistration.md` (fixed before this run).

SJ 01 jump minutes (engine unchanged) → onset second from spot 1s klines → **P1** early activity build-up, **P2** split events (bar-boundary artefact), **P3** continuation from onset+5 s to onset+65 s.

In [ ]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import signed_jumps as sj, signed_jumps_seconds as ss
for m in (sj,ss):importlib.reload(m)
assert sj.VERSION.startswith('SJ01-0.1') and ss.VERSION=='SJ04-0.1.0'
SYNTHETIC=os.getenv('SJ04_SYNTHETIC','0')=='1'
p=sj.Params();sp=ss.SecParams()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='sj04-');OUT=Path(tmp.name)/'04'
    p=sj.Params(n_boot=200,half_lives=(1,));sp=ss.SecParams(n_boot=300)
else:
    OUT=ROOT/'notebooks'/'Signed_Jumps'/'04'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
if not SYNTHETIC:
    chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_signed_jumps_seconds.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
    print(chk.stdout[-600:]);print(chk.stderr[-400:]);assert chk.returncode==0,'tests failed'
print(json.dumps(ss.registration(sp),indent=1))

## SJ 01 jumps (engine unchanged) and 1-second data

In [ ]:
T0=time.time()
if SYNTHETIC:
    secs,one=ss.simulate_seconds(n_days=14,n_assets=len(p.assets),seed=3,precursor=0.5,drift_bp=5,straddle_share=0.3)
    secs={a:v for a,v in zip(p.assets,secs.values())};one={a:v for a,v in zip(p.assets,one.values())}
    assets,pan,_=sj.prepare(one,p)
else:
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
    S={A:sj.load_spot_1m(BIN/'basis'/'spot'/f'{A}USDT_spot_1m.parquet') for A in p.assets}
    assets,pan,_=sj.prepare(S,p)
    q01=pd.read_csv(ROOT/'notebooks'/'Signed_Jumps'/'01'/'01_data_quality.csv').set_index('asset')
    for i,a in enumerate(assets):
        m=pan.asset_idx==i
        for s in sj.SIGNS:
            for per,mm in (('dev',pan.dev),('test',~pan.dev)):
                assert int(pan.J[s][m&mm].sum())==int(q01.loc[a.name,f'{s}_{per}']),(a.name,s,per)
    print('jump counts reproduce SJ 01')
    secs={}
    for A in p.assets:
        files=sorted((BIN/'spot_1s'/f'{A}USDT').glob(f'{A}USDT_spot_1s_*.parquet'))
        assert files,f'no 1s files for {A}'
        secs[A]=ss.load_spot_1s(files);print(A,len(files),'files',secs[A].index.min(),'->',secs[A].index.max(),f'{len(secs[A]):,} s')
ev=ss.jump_minutes(assets,pan,sp.cluster_gap_min)
keep=[]
for a,g in ev.groupby('asset'):
    lo,hi=secs[a].index.min()+pd.Timedelta(seconds=sp.baseline_s-sp.profile[0]),secs[a].index.max()-pd.Timedelta(seconds=sp.profile[1]+400)
    keep.append(g[(g['T']>=lo)&(g['T']<=hi)])
ev=pd.concat(keep).sort_values(['asset','T']).reset_index(drop=True)
ev=ss.onset_seconds(ev,secs)
print(f'{len(ev)} jump minutes in 1s window ({ev.starter.sum()} cluster starters); onset found for {ev.onset.notna().sum()} | {time.time()-T0:.0f}s')
desc=ev.groupby('asset').agg(jumps=('T','size'),starters=('starter','sum'),onset_found=('onset','count'),
     median_onset_u=('onset_u','median'),median_peak_share=('peak_share','median'),mean_abs_minute_bp=('minute_bp',lambda x:x.abs().mean()))
display(desc.round(2));desc.to_csv(OUT/'04_events_summary.csv')
hist=pd.cut(ev.onset_u,[0,10,20,30,40,50,60]).value_counts(normalize=True).sort_index()
print('onset second within the jump minute:');display(hist.round(3))
ev.to_csv(OUT/'04_events.csv.gz',index=False)

## P1 — activity build-up before onset

In [ ]:
act={a:ss.activity(secs[a],sp.baseline_s) for a in secs}
plc=ss.placebo_onsets(ev,secs,sp)
r1=ss.p1_test(ev,plc,secs,act,sp);print(json.dumps(r1,indent=1))
pr=ss.profile(ev,plc,secs,act,sp,bin_s=10);pr.to_csv(OUT/'04_profile_10s.csv',index=False)
pr1=ss.profile(ev,plc,secs,act,sp,bin_s=1);pr1.to_csv(OUT/'04_profile_1s.csv',index=False)
display(pr.set_index('bin_start_s').loc[[-600,-300,-180,-120,-90,-60,-40,-30,-20,-10,0,10,30,60,120,290]].round(3))

## P2 — split events (bar-boundary artefact?)

In [ ]:
pairs,r2=ss.split_pairs(ev,sp);print(json.dumps(r2,indent=1))
pairs.to_csv(OUT/'04_pairs.csv',index=False)
if len(pairs):display(pairs.groupby('same_sign').gap_s.describe().round(1))

## P3 — continuation after onset (seconds)

In [ ]:
e3=ss.post_onset(ev,secs,sp)
t3,r3=ss.p3_table(e3,sp);print(json.dumps(r3,indent=1,default=float))
t3.to_csv(OUT/'04_post_onset_grid.csv',index=False)
display(t3.pivot_table(index='delta_s',columns='h_s',values='gross_bp').round(2))
display(t3.pivot_table(index='delta_s',columns='h_s',values='share_positive').round(3))

## Decision

In [ ]:
dec={'version':ss.VERSION,'status':sp.status,'P1_early_buildup':r1,'P2_split_events':r2,'P3_continuation':r3,
     'events':int(len(ev)),'starters':int(ev.starter.sum()),'synthetic':SYNTHETIC}
print(json.dumps(dec,indent=1,default=float))
if not SYNTHETIC:
    (OUT/'04_decision.json').write_text(json.dumps(dec,indent=1,default=float))
    (OUT/'04_manifest.json').write_text(json.dumps({'registration':ss.registration(sp),'runtime_s':time.time()-T0},indent=1,default=str))
print(f'total {time.time()-T0:.0f}s')